# Lab 01: LLM API Fundamentals and Multi-turn Context (Ollama + OpenAI)

This lab is local-first for zero API cost in class.
1. Local model access via Ollama (primary)
2. OpenAI API usage (optional)


## 0) Setup

Before running:
- Install dependencies with `uv sync`
- Create `.env` from `.env.example`
- Ensure Ollama is running and your Qwen model is installed


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")

def has_real_openai_key(value: str | None) -> bool:
    if not value or not value.strip():
        return False
    cleaned = value.strip()
    lowered = cleaned.lower()
    if lowered in {"your_openai_api_key_here", "sk-your_key_here"}:
        return False
    if lowered.startswith("your_"):
        return False
    return cleaned.startswith("sk-")

OPENAI_ENABLED = has_real_openai_key(OPENAI_API_KEY)

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:8b")

print("OPENAI_MODEL (optional):", OPENAI_MODEL)
print("OPENAI enabled:", OPENAI_ENABLED)
print("OLLAMA_BASE_URL:", OLLAMA_BASE_URL)
print("OLLAMA_MODEL:", OLLAMA_MODEL)


OPENAI_MODEL (optional): gpt-5-nano
OPENAI enabled: False
OLLAMA_BASE_URL: http://localhost:11434
OLLAMA_MODEL: qwen3:8b


## How `chat.completions.create` Works

- `model`: the model name (for example `qwen3:8b` on Ollama or `gpt-5-nano` on OpenAI).
- `messages`: conversation history passed to the model.
- `temperature`: controls randomness (lower = more deterministic).

### Message Roles
- `system`: global behavior/instructions for the assistant.
- `user`: the current user request.
- `assistant`: previous model output included as conversation context.


## 1) Local Ollama (Primary): OpenAI-Compatible Qwen Call

Use this section as the default classroom path.


In [2]:
ollama_client = OpenAI(
    base_url=f"{OLLAMA_BASE_URL.rstrip('/')}/v1",
    api_key=os.getenv("OLLAMA_API_KEY", "ollama"),
)

ollama_response = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise teaching assistant."},
        {"role": "user", "content": "Explain what a token is in LLMs in 2 sentences."}
    ],
    temperature=1,
)

display(Markdown("## Qwen Response\n\n" + ollama_response.choices[0].message.content))


## Qwen Response

Tokens are the fundamental units of text processed by large language models (LLMs), such as words, subwords, or characters. They enable the model to analyze and generate language by breaking down input into manageable parts for computation.

## 2) OpenAI API (Optional)

Use this only if you want to spend on hosted/premium models.


In [ ]:
if not OPENAI_ENABLED:
    display(Markdown("**Skipped:** set a real `OPENAI_API_KEY` in `.env` to run this section."))
else:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    openai_response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "You are a concise teaching assistant."},
            {"role": "user", "content": "Explain what a token is in LLMs in 2 sentences."}
        ],
        temperature=1,
    )
    display(Markdown("## OpenAI Response\n\n" + openai_response.choices[0].message.content))


## 3) Compare Responses

Use the same prompt and compare local vs cloud output quality, style, and latency.


In [ ]:
shared_prompt = "Give 3 practical tips to write better prompts for beginner LLM users."

ollama_compare = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": shared_prompt}],
    temperature=1,
)

display(Markdown("## Ollama (Qwen)\n\n" + ollama_compare.choices[0].message.content))

if OPENAI_ENABLED:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    openai_compare = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": shared_prompt}],
        temperature=1,
    )
    display(Markdown("## OpenAI\n\n" + openai_compare.choices[0].message.content))
else:
    display(Markdown("**OpenAI compare skipped** because `OPENAI_API_KEY` is not configured."))


## 4) Multi-turn Example: Feed Assistant Output Back As Context

This demonstrates how an assistant message from turn 1 is sent back in turn 2 using `role="assistant"`.


In [ ]:
seed_topic = "vector databases in RAG"

turn1 = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful instructor."},
        {"role": "user", "content": f"Generate one strong interview question about {seed_topic}. Return the question only"},
    ],
    temperature=1,
)

assistant_question = turn1.choices[0].message.content
display(Markdown("## Turn 1: Assistant-Generated Question\n\n" + assistant_question))

turn2 = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful instructor."},
        {"role": "assistant", "content": assistant_question},
        {"role": "user", "content": "Now answer that question in a concise beginner-friendly way."},
    ],
    temperature=1,
)

display(Markdown("## Turn 2: Answer Using Assistant Context\n\n" + turn2.choices[0].message.content))


## 5) Multi-turn Example (OpenAI API, Optional)

Same pattern as above, but using OpenAI API. This is optional and may incur cost.


In [ ]:
if not OPENAI_ENABLED:
    display(Markdown("**Skipped:** set a real `OPENAI_API_KEY` in `.env` to run this section."))
else:
    seed_topic = "function calling in agents"
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    openai_turn1 = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful instructor."},
            {"role": "user", "content": f"Generate one strong interview question about {seed_topic}. Return the question only"},
        ],
        temperature=1,
    )

    openai_assistant_question = openai_turn1.choices[0].message.content
    display(Markdown("## OpenAI Turn 1: Assistant-Generated Question\n\n" + openai_assistant_question))

    openai_turn2 = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful instructor."},
            {"role": "assistant", "content": openai_assistant_question},
            {"role": "user", "content": "Now answer that question in 4 short bullet points for beginners."},
        ],
        temperature=1,
    )

    display(Markdown("## OpenAI Turn 2: Answer Using Assistant Context\n\n" + openai_turn2.choices[0].message.content))


## 6) Exercises
- Build your own 3-turn loop: (1) model generates a question, (2) you pass that as an `assistant` message, (3) model answers and then proposes one follow-up question.
- Repeat the loop for 2 more turns while keeping the full `messages` history.
- Compare how the conversation quality changes when you remove prior `assistant` messages.
- Optional: run the same loop on both Ollama and OpenAI and compare results.


**Hint (persistent message history):** keep a single `messages` list, append each new `assistant` output and `user` prompt, then pass the full list again in the next call.


---
## Multi-turn three loop with role assistant


In [3]:

seed_topic = "Stock Investment for Dummies"

turn1 = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful financial advisor."},
        {"role": "user", "content": f"Generate one strong question about {seed_topic}. Return the question only"},
    ],
    temperature=1,
)

assistant_question1 = turn1.choices[0].message.content
display(Markdown("## Turn 1: Response\n\n" + assistant_question1))


turn2 = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful financial advisor."},
        {"role": "assistant", "content": assistant_question1},
        {"role": "user", "content": "Answer that question in a concise beginner-friendly way, then propose one follow-up question."},
    ],
    temperature=1,
)

assistant_answer1 = turn2.choices[0].message.content
display(Markdown("## Turn 2: Response\n\n" + assistant_answer1))


followup_question1 = assistant_answer1.split("question:")[-1].strip()


turn3 = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful financial advisor."},
        {"role": "assistant", "content": followup_question1},
        {"role": "user", "content": "Answer that question in a concise beginner-friendly way. Explicitly state this is the last turn and the conversation ends here."},
    ],
    temperature=1,
)

assistant_answer2 = turn3.choices[0].message.content
display(Markdown("## Turn 3: Response\n\n" + assistant_answer2))

## Turn 1: Response

What are the key differences between investing in individual stocks versus index funds, and which is better for a beginner?

## Turn 2: Response

**Answer:**  
Investing in **individual stocks** means buying shares of a single company, which can lead to high rewards but also high risks (e.g., company-specific losses). **Index funds** invest in a broad market index (like the S&P 500), offering diversification, lower risk, and steady growth over time. For beginners, index funds are often better because they’re easier to manage, less volatile, and require less research.  

**Follow-up question:**  
*What are some beginner-friendly index funds or platforms to start with?*

## Turn 3: Response

To start with index funds, consider low-cost ETFs like **Vanguard S&P 500 ETF (VOO)** or **iShares Core S&P 500 ETF (IVV)**. For platforms, **Vanguard** or **Fidelity** offer beginner-friendly tools with low fees. Robo-advisors like **Betterment** or **Wealthfront** can also automate investing. Always start small, use dollar-cost averaging, and diversify across asset classes. This is the last turn. 😊

---
## Multi-turn five loop with role assistant


In [4]:

seed_topic = "Stock Investment for Dummies"
num_turns = 5  

messages = [
    {"role": "system", "content": "You are a helpful financial advisor."}
]


turn1 = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=messages + [
        {"role": "user", "content": f"Generate one strong question about {seed_topic}. Return the question only"}
    ],
    temperature=1,
)
assistant_question = turn1.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_question})
display(Markdown("## Turn 1: Assistant-Generated Question\n\n" + assistant_question))


for turn in range(2, num_turns + 1):
    if turn < num_turns:  
        user_prompt = "Answer that question in a concise beginner-friendly way, then propose one follow-up interview question."
    else:
        user_prompt = "Answer that final follow-up question in a concise beginner-friendly way. Explicitly state this is the last turn and the conversation ends here."

    turn_response = ollama_client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=messages + [
            {"role": "user", "content": user_prompt}
        ],
        temperature=1,
    )

    assistant_output = turn_response.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_output})
    display(Markdown(f"## Turn {turn}: Response\n\n" + assistant_output))

## Turn 1: Assistant-Generated Question

What's the difference between buying individual stocks and investing in index funds?

## Turn 2: Response

**Answer:**  
Buying individual stocks means investing in a single company (e.g., Apple), while index funds (like the S&P 500) track a broad market basket of stocks. Individual stocks can offer higher returns but are riskier and require research. Index funds spread risk across many companies, are easier to manage, and typically have lower fees.  

**Follow-up question:**  
*How can a beginner decide whether to invest in individual stocks or index funds based on their risk tolerance and time availability?*

## Turn 3: Response

**Answer:**  
Beginners should prioritize index funds if they have low risk tolerance or limited time, as they offer diversification, lower fees, and simplicity. Individual stocks require more research, time, and risk tolerance. Start with index funds unless you’re prepared to dedicate time to learning and monitoring specific companies.  

**Follow-up question:**  
*How do I choose the right index fund for my investment goals?*

## Turn 4: Response

**Answer:**  
Choose a broad-market index (like the S&P 500) for diversification. Look for low expense ratios and trustworthy providers (e.g., Vanguard, Fidelity). Avoid niche indices unless you have a specific goal. Start simple and stay consistent.  

**Follow-up question:**  
*How can I start investing in index funds with a small amount of money?*

## Turn 5: Response

To start investing in index funds with small money:  
1. Choose low-cost funds (e.g., Vanguard’s S&P 500 ETF or Fidelity’s index funds).  
2. Use a brokerage account (like Robinhood, Webull, or Acorns) that lets you invest small amounts.  
3. Start with monthly contributions (e.g., $50) and let time grow your money.  

This is the last turn. Let me know if you need help with anything else! 😊

--- 
## Multiturn without role assistant

In [5]:
seed_topic = "Stock Investment for Dummies"
num_turns = 5  


turn1 = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful financial advisor."},
        {"role": "user", "content": f"Generate one strong question about {seed_topic}. Return the question only"},
    ],
    temperature=1,
)
assistant_question = turn1.choices[0].message.content
display(Markdown("## Turn 1: Assistant-Generated Question\n\n" + assistant_question))


for turn in range(2, num_turns + 1):
    if turn < num_turns:
        user_prompt = "Answer that question in a concise beginner-friendly way, then propose one follow-up question."
    else:
        user_prompt = "Answer that final follow-up question in a concise beginner-friendly way. Explicitly state this is the last turn and the conversation ends here."

    turn_response = ollama_client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful financial advisor."},
            {"role": "user", "content": user_prompt}
        ],
        temperature=1,
    )

    assistant_output = turn_response.choices[0].message.content
    display(Markdown(f"## Turn {turn}: Response\n\n" + assistant_output))

## Turn 1: Assistant-Generated Question

What are the fundamental concepts every beginner needs to understand before investing in stocks?

## Turn 2: Response

**Answer:**  
A 401(k) is a retirement account where your employer offers to invest a portion of your paycheck, often with tax advantages. Contributions grow tax-deferred, and you can withdraw funds after age 59½ without penalties.  

**Follow-up Question:**  
What’s the difference between a Roth IRA and a traditional IRA?

## Turn 3: Response

**Answer:**  
To start investing, begin with small, low-cost index funds or ETFs, set regular contributions, and diversify across asset classes. Keep it simple and consistent!  

**Follow-up Question:**  
How can I determine my risk tolerance to choose the right investments?

## Turn 4: Response

It seems the question you're asking isn't included in your message. Could you please share the specific question you'd like help with? Once you do, I'll provide a clear, beginner-friendly answer and suggest a follow-up question.

## Turn 5: Response

Thanks for your question! To summarize: always prioritize your financial goals, stay informed, and consult professionals for advice. This is the last turn—feel free to reach out if you need more help later. Have a great day! 😊

---

---
## Multiturn with and without role assistant comparison

In [6]:

seed_topic = "Stock Investment for Dummies"

def run_loop(num_turns=5, keep_history=True):
    messages = [{"role": "system", "content": "You are a helpful financial advisor."}]
    outputs = []

  
    turn1 = ollama_client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=messages + [
            {"role": "user", "content": f"Generate one strong question about {seed_topic}. Return the question only"}
        ],
        temperature=1,
    )
    assistant_question = turn1.choices[0].message.content
    outputs.append(("Turn 1", assistant_question))
    if keep_history:
        messages.append({"role": "assistant", "content": assistant_question})


    for turn in range(2, num_turns + 1):
        if turn < num_turns:
            user_prompt = "Answer that question in a concise beginner-friendly way, then propose one follow-up question."
        else:
            user_prompt = "Answer that final follow-up question in a concise beginner-friendly way. Explicitly state this is the last turn and the conversation ends here."

        turn_response = ollama_client.chat.completions.create(
            model=OLLAMA_MODEL,
            messages=(messages if keep_history else [{"role": "system", "content": "You are a helpful financial advisor."}]) + [
                {"role": "user", "content": user_prompt}
            ],
            temperature=1,
        )
        assistant_output = turn_response.choices[0].message.content
        outputs.append((f"Turn {turn}", assistant_output))

        if keep_history:
            messages.append({"role": "assistant", "content": assistant_output})

    return outputs


with_history = run_loop(num_turns=5, keep_history=True)
without_history = run_loop(num_turns=5, keep_history=False)


table_md = "## Comparison: With vs Without Assistant History\n\n"
table_md += "| Turn | With History | Without History |\n"
table_md += "|------|--------------|-----------------|\n"

for i in range(len(with_history)):
    turn_label = with_history[i][0]
    with_text = with_history[i][1].replace("\n", " ")
    without_text = without_history[i][1].replace("\n", " ")
    table_md += f"| {turn_label} | {with_text} | {without_text} |\n"

display(Markdown(table_md))

## Comparison: With vs Without Assistant History

| Turn | With History | Without History |
|------|--------------|-----------------|
| Turn 1 | What are the key factors to consider when choosing individual stocks for your investment portfolio? | What's the difference between investing in individual stocks and mutual funds, and which is better for a beginner? |
| Turn 2 | **Key factors to consider when choosing individual stocks:**   1. **Company fundamentals**: Look at revenue, profits, growth potential, and financial ratios (e.g., P/E, P/B).   2. **Industry and competition**: Assess the company’s position in its market and industry trends.   3. **Management quality**: Strong leadership can drive long-term success.   4. **Valuation**: Check if the stock is overpriced or undervalued relative to its earnings.   5. **Diversification**: Avoid putting too much money into one stock.   6. **Risk tolerance**: Match your investment goals (e.g., growth vs. income).    **Follow-up question:** *How can I assess a company’s financial health quickly when researching stocks?* | **Answer:** To start investing, set clear financial goals, choose an account type (like a brokerage or retirement account), research investments, and begin with small amounts. Diversify your portfolio to reduce risk.    **Follow-up Question:** How do I choose the right investment account for my goals? |
| Turn 3 | **Quick way to assess a company’s financial health:**   Check **revenue** (is it growing?), **net profit** (is it profitable?), and the **P/E ratio** (is the stock over- or undervalued?). Also, look at **debt levels** (too much debt can be risky) and **cash flow** (does the company generate enough to sustain itself?).    **Follow-up question:** *How can I use these metrics to compare different stocks effectively?* | Sure! Please share your question, and I’ll explain it simply and clearly. After that, I’ll follow up with a related question to help you learn more. 😊    **Example follow-up question:**   *How might your answer change if you had different financial goals, like buying a home or retiring early?*    What would you like to know? |
| Turn 4 | **Concise Answer:**   Compare stocks by checking:   - **Revenue growth** (expanding business?),   - **Net profit** (is it profitable?),   - **P/E ratio** (is it over/undervalued?),   - **Debt levels** (avoid high-risk debt),   - **Cash flow** (can it sustain operations?).    Use these to evaluate which stocks align with your goals and risk tolerance.    **Follow-up Question:**   How do I adjust for differences in industries when comparing P/E ratios? | **Answer:** To start investing, set clear goals (like retirement or buying a home), choose low-cost index funds for simplicity, invest regularly via automatic transfers, and keep learning about markets.    **Follow-up Question:** How can I balance risk and returns when choosing investments? |
| Turn 5 | **Answer:**   To adjust for industry differences in P/E ratios, compare a stock’s P/E to the **average P/E of its industry** (e.g., tech vs. utilities). This shows if the stock is overvalued or undervalued relative to its peers.    **This is the last turn. The conversation ends here.** | This is the final response. If you have any more questions, feel free to ask, but this will be the last turn. The conversation ends here. |
